# Install MAFFT and TrimAl

In [ ]:
#Install MAFFT from built in repo
!apt-get -qq install mafft
!mafft --version

In [ ]:
#Install TrimAl from author's github repo and unzip
!wget -q https://github.com/inab/trimal/releases/download/v1.5.0/trimAl_Linux_x86-64.zip
!unzip -q trimAl_Linux_x86-64.zip

#Move trimal to PATH location for easier use then cleanup files we don't need
!mv trimAl_Linux_x86-64/trimal /usr/local/bin
!rm trimAl_Linux_x86-64.zip
!rm -rf trimAl_Linux_x86-64

In [ ]:
# install biopython
!pip -q install biopython pillow

In [ ]:
#Check installs worked by getting version info
!mafft --version
!trimal --version
import Bio
print (Bio.__version__)

#Run MAFFT in auto mode
* **Assumes you have uploaded the P450.fasta file to this Runtime**
* MAFFT manual: https://mafft.cbrc.jp/alignment/software/manual/manual.html

In [ ]:
# run mafft in auto mode (let mafft decided the optimal algorithm for given dataset)
!mafft --auto P450.fasta > P450_aln.fasta

# Make a dot plot of the top 16 alignments
* 16 is arbitrary - (4x4 grid)

In [ ]:
# Import tools
from google.colab import files
import numpy as np
import matplotlib.pyplot as plt
from Bio import AlignIO
from PIL import Image
import os

msa_path = "P450_aln.fasta"   # alignment file already in /content/
msa_format = "fasta"

# ---------- params you can tweak ----------
k = 9                    # k-mer size for dot plots (larger -> cleaner diagonals)
tiles_per_row = 4
tiles_per_col = 4
tile_size_px = (700, 700)   # pixel size of each tile in the final grid
out_dir = "dotplot_tiles"
grid_out = "top16_dotplots_grid.png"
# ------------------------------------------

# Load alignment
aln = AlignIO.read(msa_path, msa_format)
N = len(aln)
ids = [rec.id for rec in aln]
seqs = [str(rec.seq) for rec in aln]
seq_bytes = [np.frombuffer(s.encode("ascii"), dtype="S1") for s in seqs]

# Pairwise identity ignoring gaps (and only comparing aligned positions)
pairs = []
for i in range(N):
    A = seq_bytes[i]
    for j in range(i+1, N):
        B = seq_bytes[j]
        valid = (A != b'-') & (B != b'-')
        denom = int(valid.sum())
        pid = float((A[valid] == B[valid]).sum())/denom if denom else 0.0
        pairs.append((pid, i, j))

# Top 16 pairs
pairs.sort(key=lambda x: x[0], reverse=True)
top_pairs = pairs[:tiles_per_row * tiles_per_col]

# Helper: dot-plot points from UNGAPPED sequences using k-mer matches
def dotplot_points(s1, s2, k):
    s1u = s1.replace('-', '')
    s2u = s2.replace('-', '')
    index = {}
    # index k-mers of s2
    L2 = len(s2u)
    for j in range(L2 - k + 1):
        km = s2u[j:j+k]
        index.setdefault(km, []).append(j)
    xs, ys = [], []
    L1 = len(s1u)
    for i in range(L1 - k + 1):
        km = s1u[i:i+k]
        js = index.get(km)
        if js:
            xs.extend(js)
            ys.extend([i]*len(js))
    return np.array(xs), np.array(ys), s1u, s2u

# Make individual tiles
os.makedirs(out_dir, exist_ok=True)
tile_paths = []
for idx, (pid, i, j) in enumerate(top_pairs, start=1):
    x, y, s1u, s2u = dotplot_points(seqs[i], seqs[j], k)
    fig = plt.figure(figsize=(tile_size_px[0]/100, tile_size_px[1]/100), dpi=100)
    plt.scatter(x, y, s=0.5)   # don't set explicit colors; keep default
    plt.xlabel(ids[j])
    plt.ylabel(ids[i])
    plt.title(f"Pair {idx}: %ID={pid*100:.1f} (k={k})")
    plt.tight_layout()
    out_path = os.path.join(out_dir, f"dotpair_{idx:02d}.png")
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    tile_paths.append(out_path)

# Stitch into a 4×4 grid
grid_w = tiles_per_row * tile_size_px[0]
grid_h = tiles_per_col * tile_size_px[1]
grid = Image.new("RGB", (grid_w, grid_h), (255, 255, 255))

for idx, path in enumerate(tile_paths):
    r = idx // tiles_per_row
    c = idx % tiles_per_row
    img = Image.open(path).resize(tile_size_px)
    grid.paste(img, (c*tile_size_px[0], r*tile_size_px[1]))

grid.save(grid_out)
print("Saved grid ->", grid_out)

#Display the graphs
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=grid_out))
from google.colab import files
files.download(grid_out)

# Run TrimAl to automate trimming alignments
* https://trimal.cgenomics.org/


In [ ]:
# run trimal, output the trimmed sequences and html to visualize conserved sites
# creates P450_trimmed.fasta - for input to new MSA
!trimal -in P450_aln.fasta -out P450_trimmed.fasta -htmlout trimmed.html -automated1

#Run MAFFT on trimmed sequences

In [ ]:
# run mafft in auto mode
!mafft --auto P450_trimmed.fasta > P450_trimmed_aln.fasta

# Generate dotplots for new trimmed MSA

In [ ]:
msa_path = "P450_trimmed_aln.fasta"   # trimmed alignment file already in /content/
msa_format = "fasta"

# ---------- params you can tweak ----------
k = 9                    # k-mer size for dot plots (larger -> cleaner diagonals)
tiles_per_row = 4
tiles_per_col = 4
tile_size_px = (700, 700)   # pixel size of each tile in the final grid
out_dir = "dotplot_tiles_trimmed"
grid_out = "top16_dotplots_trimmed_grid.png"
# ------------------------------------------

# Load alignment
aln = AlignIO.read(msa_path, msa_format)
N = len(aln)
ids = [rec.id for rec in aln]
seqs = [str(rec.seq) for rec in aln]
seq_bytes = [np.frombuffer(s.encode("ascii"), dtype="S1") for s in seqs]

# Pairwise identity ignoring gaps (and only comparing aligned positions)
pairs = []
for i in range(N):
    A = seq_bytes[i]
    for j in range(i+1, N):
        B = seq_bytes[j]
        valid = (A != b'-') & (B != b'-')
        denom = int(valid.sum())
        pid = float((A[valid] == B[valid]).sum())/denom if denom else 0.0
        pairs.append((pid, i, j))

# Top 16 pairs
pairs.sort(key=lambda x: x[0], reverse=True)
top_pairs = pairs[:tiles_per_row * tiles_per_col]

# Helper: dot-plot points from UNGAPPED sequences using k-mer matches
def dotplot_points(s1, s2, k):
    s1u = s1.replace('-', '')
    s2u = s2.replace('-', '')
    index = {}
    # index k-mers of s2
    L2 = len(s2u)
    for j in range(L2 - k + 1):
        km = s2u[j:j+k]
        index.setdefault(km, []).append(j)
    xs, ys = [], []
    L1 = len(s1u)
    for i in range(L1 - k + 1):
        km = s1u[i:i+k]
        js = index.get(km)
        if js:
            xs.extend(js)
            ys.extend([i]*len(js))
    return np.array(xs), np.array(ys), s1u, s2u

# Make individual tiles
os.makedirs(out_dir, exist_ok=True)
tile_paths = []
for idx, (pid, i, j) in enumerate(top_pairs, start=1):
    x, y, s1u, s2u = dotplot_points(seqs[i], seqs[j], k)
    fig = plt.figure(figsize=(tile_size_px[0]/100, tile_size_px[1]/100), dpi=100)
    plt.scatter(x, y, s=0.5)   # don't set explicit colors; keep default
    plt.xlabel(ids[j])
    plt.ylabel(ids[i])
    plt.title(f"Pair {idx}: %ID={pid*100:.1f} (k={k})")
    plt.tight_layout()
    out_path = os.path.join(out_dir, f"dotpair_{idx:02d}.png")
    plt.savefig(out_path, bbox_inches="tight")
    plt.close(fig)
    tile_paths.append(out_path)

# Stitch into a 4×4 grid
grid_w = tiles_per_row * tile_size_px[0]
grid_h = tiles_per_col * tile_size_px[1]
grid = Image.new("RGB", (grid_w, grid_h), (255, 255, 255))

for idx, path in enumerate(tile_paths):
    r = idx // tiles_per_row
    c = idx % tiles_per_row
    img = Image.open(path).resize(tile_size_px)
    grid.paste(img, (c*tile_size_px[0], r*tile_size_px[1]))

grid.save(grid_out)
print("Saved grid ->", grid_out)

#Display the graphs
from IPython.display import Image as IPyImage, display
display(IPyImage(filename=grid_out))
from google.colab import files
files.download(grid_out)

# Create a gene tree to show relationships among the sequences
* This code creates a circular layout tree, but that is an aesthetic choice

In [ ]:
import numpy as np
from Bio import AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor, DistanceMatrix
import math
import matplotlib.pyplot as plt

# Load alignment
aln = AlignIO.read(msa_path, msa_format)
ids = [rec.id for rec in aln]
seqs = [np.frombuffer(str(rec.seq).encode("ascii"), dtype="S1") for rec in aln]
N = len(seqs)
print(f"{N} sequences; alignment length {aln.get_alignment_length()}")

# Fast p-distance: 1 - identity, comparing only positions where both are not gaps ('-')
def pairwise_pdist(A, B):
    valid = (A != b'-') & (B != b'-')
    denom = int(valid.sum())
    if denom == 0:
        return 1.0
    return 1.0 - float((A[valid] == B[valid]).sum()) / denom

# Build lower-triangular distance matrix
dm_mat = [[0.0]*(i+1) for i in range(N)]
for i in range(N):
    for j in range(i):
        dm_mat[i][j] = pairwise_pdist(seqs[i], seqs[j])

dm = DistanceMatrix(names=ids, matrix=dm_mat)

# Neighbor-Joining tree
constructor = DistanceTreeConstructor()
tree = constructor.nj(dm)

# Midpoint root for symmetry
try:
    tree.root_at_midpoint()
except Exception:
    pass

# Save Newick
newick_path = "gene_tree_nj.newick"
Phylo.write(tree, newick_path, "newick")
print("Saved Newick ->", newick_path)

# Compute clade depths (distance from root)
def clade_depths(clade, current=0.0, depths=None):
    if depths is None:
        depths = {}
    depths[clade] = current
    for child in clade.clades:
        bl = child.branch_length or 0.0
        clade_depths(child, current + bl, depths)
    return depths

depths = clade_depths(tree.root)
max_depth = max(depths.values()) or 1.0
def radius_of(clade):
    # map to [0.1, 1.0] for margin
    return 0.1 + 0.9 * (depths[clade] / max_depth)

# Evenly spaced leaf angles
terms = tree.get_terminals()
n_leaves = len(terms)
leaf_to_angle = {t: 2*math.pi * i / n_leaves for i, t in enumerate(terms)}

# Angle of internal nodes = circular mean of child angles
def compute_angles(clade):
    if clade.is_terminal():
        return leaf_to_angle[clade]
    child_angles = [compute_angles(c) for c in clade.clades]
    x = sum(math.cos(a) for a in child_angles) / len(child_angles)
    y = sum(math.sin(a) for a in child_angles) / len(child_angles)
    return math.atan2(y, x) % (2*math.pi)

angles = {}
def assign_angles(clade):
    angles[clade] = compute_angles(clade)
    for c in clade.clades:
        assign_angles(c)
assign_angles(tree.root)

# Draw on polar axes (one chart, no explicit colors)
fig = plt.figure(figsize=(12, 12))
ax = plt.subplot(111, projection="polar")
ax.set_axis_off()
# ----- choose which "level" to color by -----
L = 3  # e.g., root=0, children=1, etc.

# Compute topological levels (edge-count from root)
def clade_levels(clade, level=0, out=None):
    if out is None: out = {}
    out[clade] = level
    for ch in clade.clades:
        clade_levels(ch, level+1, out)
    return out

levels = clade_levels(tree.root)

# Anchor nodes at level L
anchors = [c for c, lv in levels.items() if lv == L]

# Assign colors and paint entire subtrees below each anchor
cmap = plt.get_cmap("tab10")
edge_colors = {}                         # (parent, child) -> color
anchor_color = {}                        # node at level L -> color

def paint_subtree(node, color):
    for ch in node.clades:
        edge_colors[(node, ch)] = color
        paint_subtree(ch, color)

for i, a in enumerate(anchors):
    col = cmap(i % 10)
    anchor_color[a] = col
    paint_subtree(a, col)

# (optional) make leaves’ label text match their subtree color
anchor_set = set(anchors)
def color_of_leaf(leaf):
    # path from root to leaf; first anchor we encounter decides color
    path = [tree.root] + tree.get_path(leaf)
    for n in path:
        if n in anchor_set:
            return anchor_color[n]
    return "0.5"


def draw_branch(parent, child):
    rp, rc = radius_of(parent), radius_of(child)
    ap, ac = angles[parent], angles[child]
    da = (ac - ap + math.pi) % (2*math.pi) - math.pi
    th = np.linspace(ap, ap + da, 24)

    col = edge_colors.get((parent, child), "0.5")
    ax.plot([ap, ap], [rp, rc], linewidth=0.7, color=col)
    ax.plot(th, np.full_like(th, rc), linewidth=0.7, color=col)

    # shortest arc to child angle at child radius
    da = (ac - ap + math.pi) % (2*math.pi) - math.pi
    th = np.linspace(ap, ap + da, 24)
    ax.plot(th, np.full_like(th, rc), linewidth=0.7)

for parent in tree.find_clades(order="level"):
    for child in parent.clades:
        draw_branch(parent, child)

# Leaf labels
label_r = 1.03
for t in terms:
    a = angles[t]
    ax.text(a, label_r, t.name, fontsize=6,
            rotation=np.degrees(a), rotation_mode="anchor",
            ha="left" if 0 <= a <= math.pi else "right",
            va="center",
            color=color_of_leaf(t))

plt.title("Neighbor-Joining gene tree (circular, p-distance)", pad=20)
png_path = "gene_tree_circular.png"
plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()
print("Saved PNG ->", png_path)

### Comparing MAFFT to MUSCLE results

In [ ]:
# Step 1: Install required dependencies
!pip install biopython pandas matplotlib seaborn opencv-python pillow -q

import os
import io
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from Bio import SeqIO
from google.colab import files

# Set plot style
sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'

print("=== STEP 1: UPLOAD YOUR FILES ===")
print("Please upload your `maxcc.afa` file first:")
afa_upload = files.upload()
afa_filename = list(afa_upload.keys())[0]

print("\nNow upload your image files (JPG, PNG, etc.):")
image_uploads = files.upload()
image_filenames = list(image_uploads.keys())

# --- FEATURE EXTRACTION FUNCTIONS ---

def extract_sequence_features(afa_file):
    """Extracts structural and composition metrics from an aligned FASTA file."""
    records = list(SeqIO.parse(afa_file, "fasta"))
    seq_data = []

    for rec in records:
        seq_str = str(rec.seq).upper()
        length = len(seq_str)
        gaps = seq_str.count('-') + seq_str.count('.')
        gap_pct = (gaps / length) * 100 if length > 0 else 0

        # Calculate GC Content if nucleotide, or Hydrophobicity index placeholder if amino acid
        a_cnt, c_cnt, g_cnt, t_cnt = seq_str.count('A'), seq_str.count('C'), seq_str.count('G'), seq_str.count('T')
        total_bases = a_cnt + c_cnt + g_cnt + t_cnt
        gc_content = ((g_cnt + c_cnt) / total_bases * 100) if total_bases > 0 else 0

        # Calculate sequence complexity via Shannon Entropy
        char_counts = pd.Series(list(seq_str)).value_counts()
        probs = char_counts / len(seq_str)
        entropy = -sum(probs * np.log2(probs))

        seq_data.append({
            'ID': rec.id,
            'Type': 'Sequence',
            'Length': length,
            'Gap_%': gap_pct,
            'Primary_Metric (GC/Comp)': gc_content,
            'Complexity_Entropy': entropy
        })
    return pd.DataFrame(seq_data)

def extract_image_features(image_dict):
    """Extracts spatial, color, and complexity metrics from uploaded images."""
    img_data = []

    for fname, content in image_dict.items():
        try:
            img = Image.open(io.BytesIO(content)).convert('RGB')
            img_np = np.array(img)

            # Dimensions
            width, height = img.size
            total_pixels = width * height

            # Brightness / Color intensity
            mean_intensity = np.mean(img_np)

            # Image Complexity / Sharpness via Laplacian Variance
            gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
            sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()

            # Entropy / Visual Complexity
            hist, _ = np.histogram(gray, bins=256, range=(0, 256), density=True)
            hist = hist[hist > 0]
            entropy = -np.sum(hist * np.log2(hist))

            img_data.append({
                'ID': fname,
                'Type': 'Image',
                'Length': total_pixels,               # Proxy for 'data size'
                'Gap_%': 0.0,                         # Blank padding equivalent
                'Primary_Metric (GC/Comp)': mean_intensity, # Brightness scale
                'Complexity_Entropy': entropy
            })
        except Exception as e:
            print(f"Skipping {fname}: Not a valid image file ({e})")

    return pd.DataFrame(img_data)

# --- EXECUTION & COMPARISON ---

print("\n=== STEP 2: PROCESSING & EXTRACTING METRICS ===")
df_seq = extract_sequence_features(afa_filename)
df_img = extract_image_features(image_uploads)

df_all = pd.concat([df_seq, df_img], ignore_index=True)

# Min-Max Normalization across unified metrics for balanced side-by-side comparison
numeric_cols = ['Length', 'Primary_Metric (GC/Comp)', 'Complexity_Entropy']
df_norm = df_all.copy()
for col in numeric_cols:
    min_val = df_norm[col].min()
    max_val = df_norm[col].max()
    df_norm[col] = (df_norm[col] - min_val) / (max_val - min_val) if max_val != min_val else 0

print("\n--- Summary Table (Raw Values) ---")
display(df_all)

# --- STEP 3: VISUAL COMPARISONS ---

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Complexity Comparison (Shannon Entropy)
sns.barplot(data=df_all, x='ID', y='Complexity_Entropy', hue='Type', ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Informational Complexity (Shannon Entropy)', fontsize=12, fontweight='bold')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. Normalized Profile Heatmap
sns.heatmap(
    df_norm.set_index('ID')[numeric_cols],
    annot=True,
    cmap='viridis',
    ax=axes[0, 1],
    cbar_kws={'label': 'Normalized Scale (0-1)'}
)
axes[0, 1].set_title('Normalized Multi-Modal Profile Matrix', fontsize=12, fontweight='bold')

# 3. Size/Length Distribution Comparison
sns.kdeplot(data=df_seq['Length'], label='Sequences (bp/aa)', ax=axes[1, 0], color='teal', fill=True)
axes[1, 0].set_title('Sequence Length Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Sequence Alignment Position Count')

# 4. Feature Radar/Distribution Chart for Cross-Modal Comparison
df_melted = pd.melt(df_norm, id_vars=['ID', 'Type'], value_vars=numeric_cols)
sns.boxplot(data=df_melted, x='variable', y='value', hue='Type', ax=axes[1, 1], palette='Set1')
axes[1, 1].set_title('Normalized Feature Variance: Sequence vs Images', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Feature Category')
axes[1, 1].set_ylabel('Normalized Score')

plt.tight_layout()
plt.show()

# --- STEP 3: REFINED & CLEAN VISUAL COMPARISONS ---

# Configure visual style for publication-ready outputs
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Informational Complexity (Grouped Summary instead of 100s of individual bars)
sns.boxplot(
    data=df_all,
    x='Type',
    y='Complexity_Entropy',
    palette=['#4c72b0', '#dd8452'],
    ax=axes[0, 0],
    width=0.4
)
sns.stripplot(
    data=df_all,
    x='Type',
    y='Complexity_Entropy',
    color='black',
    alpha=0.3,
    jitter=0.2,
    ax=axes[0, 0]
)
axes[0, 0].set_title('Informational Complexity Distribution (Shannon Entropy)', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Shannon Entropy')
axes[0, 0].set_xlabel('')

# 2. Aggregated Type Profile (Heatmap of Mean Metrics per Category)
df_grouped = df_all.groupby('Type')[['Length', 'Primary_Metric (GC/Comp)', 'Complexity_Entropy']].mean()
# Normalize mean values across categories for comparison
df_grouped_norm = (df_grouped - df_grouped.min()) / (df_grouped.max() - df_grouped.min() + 1e-9)

sns.heatmap(
    df_grouped_norm.T,
    annot=True,
    fmt=".2f",
    cmap='viridis',
    cbar_kws={'label': 'Relative Scale'},
    ax=axes[0, 1]
)
axes[0, 1].set_title('Mean Feature Profile: Sequences vs Images', fontsize=12, fontweight='bold')
axes[0, 1].set_yticklabels(['Length / Area', 'Primary Metric', 'Entropy'], rotation=0)

# 3. Size / Scale Distribution (Log Scale to handle disparate data sizes)
sns.histplot(
    data=df_all,
    x='Length',
    hue='Type',
    log_scale=True,
    element='step',
    stat='density',
    common_norm=False,
    palette=['#4c72b0', '#dd8452'],
    ax=axes[1, 0]
)
axes[1, 0].set_title('Data Scale Distribution (Log Base 10)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Length (bp/aa) / Image Area (pixels) [Log Scale]')
axes[1, 0].set_ylabel('Density')

# 4. Multi-Feature Comparison Boxplot with Clean Labels
df_melted = pd.melt(df_norm, id_vars=['Type'], value_vars=numeric_cols)
# Clean up variable names for x-axis display
df_melted['variable'] = df_melted['variable'].str.replace('Primary_Metric (GC/Comp)', 'Primary Metric')

sns.boxplot(
    data=df_melted,
    x='variable',
    y='value',
    hue='Type',
    palette=['#4c72b0', '#dd8452'],
    ax=axes[1, 1]
)
axes[1, 1].set_title('Normalized Feature Comparison', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Feature Category')
axes[1, 1].set_ylabel('Normalized Score (0 - 1)')
axes[1, 1].set_xticklabels(['Length', 'Primary Metric', 'Entropy'])

plt.tight_layout()
plt.show()